# EUR_PC6_DATA — graphs of the edge-type ablation PC6 (Track F, chat D37-F)

**Design:** `results/eureka_d37f/DESIGN_PC6.md` §3 (approved by the author 22/11/2026 07:32). From the raw EDF (verified
against PhysioNet in the session) the record builders rebuild every window of the 23 CHB-MIT cases (defense-time file
set, committed record statistics) and of chb24 + Siena (label-free statistics, x1 convention) with one change: per window
wPLI and AEC are computed once and three top-20 % graphs are kept — the record rule (checked against the record files,
gate G-adj), **wPLI only (11a)** and **AEC only (11b)**. A SMOKE first (chb08, first 2 files = the record, G-feat0;
projection, STOP if > 10.5 h). **Accelerator:** None (CPU). **Internet:** ON (pip, git clone, PhysioNet checksums).
**Inputs (Add Input):** `combined-siena-scalp-and-chb-mit-eeg-datasets`, `v9-x3-fold1`, `v9-x3-fold3`,
`thesis-r2-external-features`. **Output:** `/kaggle/working/eureka/pc6_data` — small files (→ `results/eureka_d37f/pc6_data`,
committed) and `graphs/` (≈ 0.3–1.1 GB of `{case}_g11.npz`, NOT committed: create the Kaggle dataset
`thesis-pc6-graphs` from this output). **Runtime:** ≈ 2.5–3.5 h (computed from EUR_9A's and R0_EXT's measured builds).
Code: `github.com/Nhanvas/Thesis` ref `main`; every module's sha256 is asserted below.
**Nothing printed here is a research result.** Do not edit; *Save Version → Save & Run All (Commit)*.

In [ ]:
# SETUP: packages + repository (read only)
import os, sys, subprocess, time, json, hashlib
T_NB = time.time()
NOTEBOOK, REPO_REF = "EUR_PC6_DATA", "main"
TEST = bool(os.environ.get("EUR_TEST_REPO"))                     # synthetic test in the build container only
WORK = os.environ.get("EUR_TEST_WORK", "/kaggle/working")
INPUT = os.environ.get("EUR_TEST_INPUT", "/kaggle/input")
SCRATCH = os.environ.get("EUR_TEST_SCRATCH", "/tmp/r2")
if TEST:
    REPO = os.environ["EUR_TEST_REPO"]; HEAD = "test"
else:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "mne==1.12.1", "torch_geometric==2.7.0", "ruptures==1.1.10", "timescoring==0.0.7"])
    REPO = "/tmp/Thesis"
    if not os.path.exists(REPO + "/src/eur_pc6_data.py"):
        subprocess.check_call(f"rm -rf {REPO} && git clone -q https://github.com/Nhanvas/Thesis.git {REPO} "
                              f"&& cd {REPO} && git checkout -q {REPO_REF}", shell=True)
    HEAD = subprocess.check_output(["git", "-C", REPO, "rev-parse", "HEAD"]).decode().strip()
print("repository", REPO, HEAD, f"({time.time()-T_NB:.0f}s)")

In [ ]:
# START: sha256 of every module (the reviewed code only), environment, imports
WANT = {"eur_pc6_data.py": "1a2bcf76d6a6218e", "eur_abl.py": "428c16c0456333db", "r2_r1.py": "76cbe51ff0180d4a", "r2_core.py": "902d15ad4feb3e8a", "e16_recording_baseline.py": "f170e1c59f5b0447", "e17b_latent_family.py": "3b4f034dca00eaf3", "p3c_freeze.py": "dabe4139f6005fd2", "p3_levers.py": "feb247d2d08c52ba", "p2_scalars.py": "11feeaf3ba95ced4", "e22_test_oneshot.py": "669350bbfaaf0f4e", "e23_test_look7.py": "88587668baf12a64", "x1_external.py": "9962e3e7fe4ddc35", "p2_dump_seeds23.py": "0bc887542989310f", "e18_cpd_localizer.py": "e18c0ffe37f78e77", "e10_kernel_cpd.py": "5c692e33d92fd0fb", "f1_core.py": "bf4bfd69d7669226", "model_cmp_core.py": "179b91278035b37a", "retrain/gae_joint.py": "06b0566c97cd0ec4", "retrain/train_gae_v2.py": "2503a0e01ac9691e", "dataprep/preprocessing.py": "0172fbea61b28540", "dataprep/graph_construction.py": "c22a976fdc531c8c", "dataprep/feature_extraction.py": "df3c01546eedc76e", "continuous_eval.py": "8fea38dabee5ad25", "c1b_build_trainset.py": "b4596a8aa3797653", "evaluation_protocol.py": "7caacac9b31bd0f7"}
sha = lambda p: hashlib.sha256(open(p, "rb").read()).hexdigest()[:16]
bad = {n: sha(f"{REPO}/src/{n}") for n in WANT if sha(f"{REPO}/src/{n}") != WANT[n]}
assert not bad, f"modules differ from the reviewed code (rebuild the notebook or pin the right commit): {bad}"
os.environ.update(THESIS_REPO=REPO, R2_WORK=WORK, R2_INPUT=INPUT, R2_SCRATCH=SCRATCH, OMP_NUM_THREADS="1",
                  OPENBLAS_NUM_THREADS="1", MKL_NUM_THREADS="1")
if TEST:
    os.environ["R2_TEST"] = "1"
sys.path[:0] = [REPO + "/src"]
import eur_pc6_data as M
assert M.__file__ == f"{REPO}/src/eur_pc6_data.py", M.__file__
print("CPU cores:", os.cpu_count(), "| eur_pc6_data", M.sha16(M.__file__), "| cases", len(M.CHB + M.EXT))

In [ ]:
# RUN: G-raw, G-data, SMOKE (G-stack, G-feat0, PROJ), CHB-MIT pool, external cases, G-stack, G-struct, G-adj, G-edges,
# G-worker, G-done
ERR = None
try:
    M.main(NOTEBOOK, HEAD, workers=int(os.environ.get("EUR_WORKERS", "0")) or None)
except SystemExit as e:
    ERR = str(e)

In [ ]:
# END: gates, per-case table, outputs, measured runtime
OUT = os.path.join(WORK, "eureka", "pc6_data")
for f in ("gates.json", "projection.json", "run_meta.json"):
    p = os.path.join(OUT, f)
    if os.path.exists(p):
        print("=====", f); print(open(p).read()[:6000])
for root, _, fs in os.walk(OUT):
    for f in sorted(fs):
        p = os.path.join(root, f); print(os.path.relpath(p, WORK), os.path.getsize(p))
print(f"MEASURED notebook runtime: {(time.time() - T_NB) / 3600:.3f} h")
if ERR:
    raise RuntimeError(f"STOPPED: {ERR}")